In [12]:
import pandas as pd
import numpy as np
import yfinance as yf
import os

print("All libraries imported successfully!")

All libraries imported successfully!


In [13]:
import yfinance as yf
import pandas as pd
import os

stocks = {
    "TCS": "TCS.NS",
    "INFOSYS": "INFY.NS",
    "RELIANCE": "RELIANCE.NS"
}

os.makedirs("data/raw", exist_ok=True)

stock_data = {}

for company, ticker in stocks.items():

    print(f"Downloading {company}...")

    df = yf.download(
        ticker,
        period="5y",
        interval="1d",
        auto_adjust=True,
        progress=False,
        multi_level_index=False
    )

    if df.empty:
        print(f"No data available for {company}")
        continue

    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)

    df = df.reset_index()
    df["Company"] = company

    df = df[
        ["Date", "Open", "High", "Low",
         "Close", "Volume", "Company"]
    ]

    df = df.dropna()
    df = df.sort_values("Date")

    stock_data[company] = df

    df.to_csv(
        f"data/raw/{company.lower()}_daily.csv",
        index=False
    )

    print(f"{company}: {len(df)} rows downloaded")

TCS: 1236 rows downloaded
INFOSYS: 1236 rows downloaded
RELIANCE: 1236 rows downloaded


In [14]:
if len(stock_data) == 3:

    all_stocks = pd.concat(
        stock_data.values(),
        ignore_index=True
    )

    all_stocks = all_stocks.sort_values(
        ["Company", "Date"]
    ).reset_index(drop=True)

    all_stocks.to_csv(
        "data/raw/all_stocks_raw.csv",
        index=False
    )

    print("All 3 stocks combined successfully!")
    print("Total rows:", len(all_stocks))

else:
    print("Warning: All 3 stocks data not available.")

All 3 stocks combined successfully!
Total rows: 3708


In [15]:
print("DATASET SHAPE:")
print(all_stocks.shape)

print("\nFIRST 5 ROWS:")
display(all_stocks.head())

print("\nCOMPANY-WISE ROW COUNT:")
print(all_stocks["Company"].value_counts())

print("\nMISSING VALUES:")
print(all_stocks.isnull().sum())

print("\nDUPLICATE ROWS:")
print(all_stocks.duplicated().sum())

print("\nDATE RANGE:")
print("Start:", all_stocks["Date"].min())
print("End:", all_stocks["Date"].max())

DATASET SHAPE:
(3708, 7)

FIRST 5 ROWS:


,Date,Open,High,Low,Close,Volume,Company
0,2021-11-15,1530.847853,1546.224215,1525.178385,1535.185913,4354074,INFOSYS
1,2021-11-16,1537.634334,1553.912600,1526.467157,1539.610107,6034233,INFOSYS
2,2021-11-17,1539.824779,1547.942415,1529.645384,1535.443726,4732492,INFOSYS
3,2021-11-18,1530.418597,1537.161832,1511.004910,1528.528809,4326924,INFOSYS
4,2021-11-22,1532.480100,1533.682740,1504.261527,1511.348389,6621839,INFOSYS



COMPANY-WISE ROW COUNT:
Company
INFOSYS     1236
RELIANCE    1236
TCS         1236
Name: count, dtype: int64

MISSING VALUES:
Date       0
Open       0
High       0
Low        0
Close      0
Volume     0
Company    0
dtype: int64

DUPLICATE ROWS:
0

DATE RANGE:
Start: 2021-11-15 00:00:00
End: 2026-11-13 00:00:00


In [16]:
def add_features(df):

    df = df.sort_values("Date").copy()

    # Daily returns
    df["Daily_Return"] = df["Close"].pct_change() * 100

    # Moving averages
    df["SMA_20"] = df["Close"].rolling(20).mean()
    df["SMA_50"] = df["Close"].rolling(50).mean()

    # Exponential moving average
    df["EMA_20"] = df["Close"].ewm(
        span=20, adjust=False
    ).mean()

    # Volatility
    df["Volatility_20"] = df["Daily_Return"].rolling(20).std()

    # RSI
    delta = df["Close"].diff()

    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)

    avg_gain = gain.rolling(14).mean()
    avg_loss = loss.rolling(14).mean()

    rs = avg_gain / avg_loss.replace(0, np.nan)

    df["RSI_14"] = 100 - (100 / (1 + rs))

    # MACD
    ema_12 = df["Close"].ewm(span=12, adjust=False).mean()
    ema_26 = df["Close"].ewm(span=26, adjust=False).mean()

    df["MACD"] = ema_12 - ema_26

    df = df.dropna().reset_index(drop=True)

    return df


feature_data = pd.concat(
    [
        add_features(group)
        for company, group in all_stocks.groupby("Company")
    ],
    ignore_index=True
)

feature_data.to_csv(
    "data/processed/stock_features.csv",
    index=False
)

print("Feature engineering completed!")
print("Dataset shape:", feature_data.shape)

display(feature_data.head())

Feature engineering completed!
Dataset shape: (3561, 14)


,Date,Open,High,Low,Close,Volume,Company,Daily_Return,SMA_20,SMA_50,EMA_20,Volatility_20,RSI_14,MACD
0,2022-01-24,1516.158831,1519.294251,1484.375332,1491.934692,7116712,INFOSYS,-2.738413,1602.573566,1549.681067,1585.007123,1.550438,30.333134,0.688866
1,2022-01-25,1483.172788,1494.683591,1460.752555,1479.350220,9137653,INFOSYS,-0.843500,1595.450189,1548.564353,1574.944560,1.511586,33.574672,-7.585963
2,2022-01-27,1462.041248,1468.655611,1430.257743,1441.940308,11231734,INFOSYS,-2.528807,1586.561554,1546.610957,1562.277489,1.578723,32.135543,-16.966907
3,2022-01-28,1444.001921,1483.989048,1442.713401,1448.468750,10998502,INFOSYS,0.452754,1577.685803,1544.871458,1551.438561,1.580878,33.731291,-23.602507
4,2022-01-31,1475.785307,1502.844234,1473.508900,1491.419312,8788872,INFOSYS,2.965239,1571.176636,1544.129268,1545.722442,1.767135,35.936116,-25.106110


In [17]:
feature_data = feature_data.sort_values(
    ["Company", "Date"]
).copy()

# Next 5 trading days return
feature_data["Future_Return_5D"] = (
    feature_data.groupby("Company")["Close"]
    .transform(lambda x: x.shift(-5) / x - 1)
)

# 1 = Price goes UP, 0 = Price goes DOWN
feature_data["Target_Up"] = (
    feature_data["Future_Return_5D"] > 0
).astype(int)

# Remove last 5 rows per company (no future data)
feature_data = (
    feature_data.groupby("Company", group_keys=False)
    .apply(lambda x: x.iloc[:-5])
    .reset_index(drop=True)
)

feature_data.to_csv(
    "data/processed/stock_features.csv",
    index=False
)

print("Prediction targets created successfully!")
print("\nTarget distribution:")
print(feature_data["Target_Up"].value_counts())

display(feature_data.head())

Prediction targets created successfully!

Target distribution:
Target_Up
1    1810
0    1736
Name: count, dtype: int64


,Date,Open,High,Low,Close,Volume,Daily_Return,SMA_20,SMA_50,EMA_20,Volatility_20,RSI_14,MACD,Future_Return_5D,Target_Up
0,2022-01-24,1516.158831,1519.294251,1484.375332,1491.934692,7116712,-2.738413,1602.573566,1549.681067,1585.007123,1.550438,30.333134,0.688866,0.020296,1
1,2022-01-25,1483.172788,1494.683591,1460.752555,1479.350220,9137653,-0.843500,1595.450189,1548.564353,1574.944560,1.511586,33.574672,-7.585963,0.037714,1
2,2022-01-27,1462.041248,1468.655611,1430.257743,1441.940308,11231734,-2.528807,1586.561554,1546.610957,1562.277489,1.578723,32.135543,-16.966907,0.035714,1
3,2022-01-28,1444.001921,1483.989048,1442.713401,1448.468750,10998502,0.452754,1577.685803,1544.871458,1551.438561,1.580878,33.731291,-23.602507,0.032558,1
4,2022-01-31,1475.785307,1502.844234,1473.508900,1491.419312,8788872,2.965239,1571.176636,1544.129268,1545.722442,1.767135,35.936116,-25.106110,-0.012211,0


In [18]:
import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

# Prepare data
model_data = feature_data.copy()
model_data["Date"] = pd.to_datetime(model_data["Date"])

# Features and target
X = model_data.drop(
    columns=["Date", "Company", "Target_Up", "Future_Return_5D"],
    errors="ignore"
)

# Encode company name
company_encoded = pd.get_dummies(
    model_data["Company"],
    prefix="Company",
    dtype=int
)

X = pd.concat([X, company_encoded], axis=1)
y = model_data["Target_Up"]

# Chronological train-test split
cutoff = model_data["Date"].quantile(0.80)

train_mask = model_data["Date"] < cutoff
test_mask = model_data["Date"] >= cutoff

X_train = X.loc[train_mask]
X_test = X.loc[test_mask]
y_train = y.loc[train_mask]
y_test = y.loc[test_mask]

# Train model
rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    min_samples_split=10,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

# Predictions
rf_predictions = rf_model.predict(X_test)

# Evaluation
accuracy = accuracy_score(y_test, rf_predictions)

print("RANDOM FOREST MODEL RESULTS")
print("---------------------------")
print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))
print("Accuracy:", round(accuracy * 100, 2), "%")

print("\nClassification Report:")
print(classification_report(y_test, rf_predictions))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, rf_predictions))

KeyError: 'Company'

In [19]:
print("feature_data columns:")
print(feature_data.columns.tolist())

print("\nDataset shape:")
print(feature_data.shape)

print("\nCompany column exists?")
print("Company" in feature_data.columns)

print("\nFirst 5 rows:")
display(feature_data.head())

feature_data columns:
['Date', 'Open', 'High', 'Low', 'Close', 'Volume', 'Daily_Return', 'SMA_20', 'SMA_50', 'EMA_20', 'Volatility_20', 'RSI_14', 'MACD', 'Future_Return_5D', 'Target_Up']

Dataset shape:
(3546, 15)

Company column exists?
False

First 5 rows:


,Date,Open,High,Low,Close,Volume,Daily_Return,SMA_20,SMA_50,EMA_20,Volatility_20,RSI_14,MACD,Future_Return_5D,Target_Up
0,2022-01-24,1516.158831,1519.294251,1484.375332,1491.934692,7116712,-2.738413,1602.573566,1549.681067,1585.007123,1.550438,30.333134,0.688866,0.020296,1
1,2022-01-25,1483.172788,1494.683591,1460.752555,1479.350220,9137653,-0.843500,1595.450189,1548.564353,1574.944560,1.511586,33.574672,-7.585963,0.037714,1
2,2022-01-27,1462.041248,1468.655611,1430.257743,1441.940308,11231734,-2.528807,1586.561554,1546.610957,1562.277489,1.578723,32.135543,-16.966907,0.035714,1
3,2022-01-28,1444.001921,1483.989048,1442.713401,1448.468750,10998502,0.452754,1577.685803,1544.871458,1551.438561,1.580878,33.731291,-23.602507,0.032558,1
4,2022-01-31,1475.785307,1502.844234,1473.508900,1491.419312,8788872,2.965239,1571.176636,1544.129268,1545.722442,1.767135,35.936116,-25.106110,-0.012211,0


In [20]:
import pandas as pd
import os

# Read fresh raw datasets
files = {
    "TCS": "data/raw/tcs_daily.csv",
    "INFOSYS": "data/raw/infosys_daily.csv",
    "RELIANCE": "data/raw/reliance_daily.csv"
}

all_stocks_list = []

for company, path in files.items():
    df = pd.read_csv(path)
    df["Company"] = company
    all_stocks_list.append(df)

all_stocks = pd.concat(
    all_stocks_list,
    ignore_index=True
)

all_stocks["Date"] = pd.to_datetime(all_stocks["Date"])

# Rebuild features separately for each company
feature_list = []

for company, group in all_stocks.groupby("Company"):
    result = add_features(group)
    result["Company"] = company
    feature_list.append(result)

feature_data = pd.concat(
    feature_list,
    ignore_index=True
)

os.makedirs("data/processed", exist_ok=True)

feature_data.to_csv(
    "data/processed/stock_features.csv",
    index=False
)

print("Company column exists:", "Company" in feature_data.columns)
print("Dataset shape:", feature_data.shape)
print("\nCompany counts:")
print(feature_data["Company"].value_counts())

display(feature_data.head())

Company column exists: True
Dataset shape: (3561, 14)

Company counts:
Company
INFOSYS     1187
RELIANCE    1187
TCS         1187
Name: count, dtype: int64


,Date,Open,High,Low,Close,Volume,Company,Daily_Return,SMA_20,SMA_50,EMA_20,Volatility_20,RSI_14,MACD
0,2022-01-24,1516.158831,1519.294251,1484.375332,1491.934692,7116712,INFOSYS,-2.738413,1602.573566,1549.681067,1585.007123,1.550438,30.333134,0.688866
1,2022-01-25,1483.172788,1494.683591,1460.752555,1479.350220,9137653,INFOSYS,-0.843500,1595.450189,1548.564353,1574.944560,1.511586,33.574672,-7.585963
2,2022-01-27,1462.041248,1468.655611,1430.257743,1441.940308,11231734,INFOSYS,-2.528807,1586.561554,1546.610957,1562.277489,1.578723,32.135543,-16.966907
3,2022-01-28,1444.001921,1483.989048,1442.713401,1448.468750,10998502,INFOSYS,0.452754,1577.685803,1544.871458,1551.438561,1.580878,33.731291,-23.602507
4,2022-01-31,1475.785307,1502.844234,1473.508900,1491.419312,8788872,INFOSYS,2.965239,1571.176636,1544.129268,1545.722442,1.767135,35.936116,-25.106110


In [21]:
print("Dataset Shape:", feature_data.shape)

print("\nCompany-wise records:")
print(feature_data["Company"].value_counts())

print("\nMissing Values:")
print(feature_data.isnull().sum())

print("\nSaved file exists:",
      os.path.exists("data/processed/stock_features.csv"))

Dataset Shape: (3561, 14)

Company-wise records:
Company
INFOSYS     1187
RELIANCE    1187
TCS         1187
Name: count, dtype: int64

Missing Values:
Date             0
Open             0
High             0
Low              0
Close            0
Volume           0
Company          0
Daily_Return     0
SMA_20           0
SMA_50           0
EMA_20           0
Volatility_20    0
RSI_14           0
MACD             0
dtype: int64

Saved file exists: True


In [22]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

print("Random Forest libraries ready!")

Random Forest libraries ready!


In [23]:

import os
import joblib
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

os.makedirs("models", exist_ok=True)

features = [
    "Open", "High", "Low", "Close", "Volume",
    "Daily_Return", "SMA_20", "SMA_50",
    "EMA_20", "Volatility_20", "RSI_14", "MACD"
]

rf_results = {}

for company in ["TCS", "INFOSYS", "RELIANCE"]:

    print(f"\nTraining Random Forest: {company}")

    df = feature_data[
        feature_data["Company"] == company
    ].copy()

    df = df.sort_values("Date").reset_index(drop=True)

    # Target: price direction after 5 trading days
    df["Future_Close"] = df["Close"].shift(-5)
    df["Target"] = (
        df["Future_Close"] > df["Close"]
    ).astype(int)

    df = df.iloc[:-5].copy()

    X = df[features]
    y = df["Target"]

    # Chronological 80/20 split with 5-row gap
    split = int(len(df) * 0.8)

    X_train = X.iloc[:split - 5]
    y_train = y.iloc[:split - 5]

    X_test = X.iloc[split:]
    y_test = y.iloc[split:]

    model = RandomForestClassifier(
        n_estimators=200,
        max_depth=10,
        min_samples_split=10,
        random_state=42,
        class_weight="balanced",
        n_jobs=-1
    )

    model.fit(X_train, y_train)

    predictions = model.predict(X_test)

    accuracy = accuracy_score(y_test, predictions)

    print("Accuracy:", round(accuracy * 100, 2), "%")
    print("\nClassification Report:")
    print(
        classification_report(
            y_test,
            predictions,
            zero_division=0
        )
    )

    joblib.dump(
        model,
        f"models/{company.lower()}_random_forest.pkl"
    )

    rf_results[company] = accuracy

print("\nAll Random Forest models trained!")



Training Random Forest: TCS
Accuracy: 44.3 %

Classification Report:
              precision    recall  f1-score   support

           0       0.63      0.19      0.29       143
           1       0.40      0.83      0.54        94

    accuracy                           0.44       237
   macro avg       0.51      0.51      0.42       237
weighted avg       0.54      0.44      0.39       237


Training Random Forest: INFOSYS
Accuracy: 51.9 %

Classification Report:
              precision    recall  f1-score   support

           0       0.54      0.31      0.39       120
           1       0.51      0.74      0.60       117

    accuracy                           0.52       237
   macro avg       0.53      0.52      0.50       237
weighted avg       0.53      0.52      0.50       237


Training Random Forest: RELIANCE
Accuracy: 48.52 %

Classification Report:
              precision    recall  f1-score   support

           0       0.53      0.66      0.59       131
           1     

In [25]:
import xgboost as xgb

print("XGBoost installed successfully!")
print("Version:", xgb.__version__)

XGBoost installed successfully!
Version: 3.4.1


In [26]:

import os
import joblib
import numpy as np

from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

os.makedirs("models", exist_ok=True)

xgb_results = {}

for company in ["TCS", "INFOSYS", "RELIANCE"]:

    print(f"\nTraining XGBoost: {company}")

    df = feature_data[
        feature_data["Company"] == company
    ].copy()

    df = df.sort_values("Date").reset_index(drop=True)

    # Predict closing price after 5 trading days
    df["Target"] = df["Close"].shift(-5)
    df = df.iloc[:-5].copy()

    X = df[features]
    y = df["Target"]

    # Chronological split
    split = int(len(df) * 0.8)

    X_train = X.iloc[:split - 5]
    y_train = y.iloc[:split - 5]

    X_test = X.iloc[split:]
    y_test = y.iloc[split:]

    model = XGBRegressor(
        n_estimators=500,
        learning_rate=0.03,
        max_depth=4,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="reg:squarederror",
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_train, y_train)

    predictions = model.predict(X_test)

    mae = mean_absolute_error(y_test, predictions)
    rmse = np.sqrt(mean_squared_error(y_test, predictions))
    mape = np.mean(
        np.abs((y_test - predictions) / y_test)
    ) * 100

    print("MAE:", round(mae, 2))
    print("RMSE:", round(rmse, 2))
    print("MAPE:", round(mape, 2), "%")

    xgb_results[company] = {
        "MAE": mae,
        "RMSE": rmse,
        "MAPE": mape
    }

    # Refit final model using all available labelled data
    final_model = XGBRegressor(
        n_estimators=500,
        learning_rate=0.03,
        max_depth=4,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="reg:squarederror",
        random_state=42,
        n_jobs=-1
    )

    final_model.fit(X, y)

    joblib.dump(
        final_model,
        f"models/{company.lower()}_xgboost.pkl"
    )

print("\nAll XGBoost models trained successfully!")



Training XGBoost: TCS
MAE: 354.68
RMSE: 409.32
MAPE: 15.61 %

Training XGBoost: INFOSYS
MAE: 72.07
RMSE: 88.24
MAPE: 6.33 %

Training XGBoost: RELIANCE
MAE: 44.08
RMSE: 55.53
MAPE: 3.22 %

All XGBoost models trained successfully!


In [27]:
import tensorflow as tf

print("TensorFlow version:", tf.__version__)
print("LSTM environment ready!")

TensorFlow version: 2.21.0
LSTM environment ready!


In [28]:

import os
import numpy as np
import joblib
import tensorflow as tf

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

os.makedirs("models", exist_ok=True)

lstm_results = {}
lstm_scalers = {}

LOOKBACK = 60
HORIZON = 5

for company in ["TCS", "INFOSYS", "RELIANCE"]:

    print(f"\nTraining LSTM: {company}")

    df = feature_data[
        feature_data["Company"] == company
    ].copy()

    df = df.sort_values("Date").reset_index(drop=True)

    df["Target"] = df["Close"].shift(-HORIZON)
    df = df.iloc[:-HORIZON].reset_index(drop=True)

    split = int(len(df) * 0.8)

    # Fit feature scaler only on training period
    feature_scaler = MinMaxScaler()
    feature_scaler.fit(df[features].iloc[:split])

    X_scaled = feature_scaler.transform(df[features])

    # Fit target scaler only on training labels
    train_target_indices = np.arange(
        LOOKBACK - 1, split - HORIZON
    )

    target_scaler = MinMaxScaler()
    target_scaler.fit(
        df["Target"].iloc[train_target_indices].values.reshape(-1, 1)
    )

    y_scaled = target_scaler.transform(
        df["Target"].values.reshape(-1, 1)
    )

    X_train, y_train = [], []
    X_test, y_test = [], []

    for i in range(LOOKBACK - 1, len(df)):

        sequence = X_scaled[i - LOOKBACK + 1:i + 1]
        target = y_scaled[i]

        if i < split - HORIZON:
            X_train.append(sequence)
            y_train.append(target)
        elif i >= split:
            X_test.append(sequence)
            y_test.append(target)

    X_train = np.array(X_train)
    y_train = np.array(y_train)

    X_test = np.array(X_test)
    y_test = np.array(y_test)

    model = Sequential([
        LSTM(64, return_sequences=True,
             input_shape=(LOOKBACK, len(features))),
        Dropout(0.2),
        LSTM(32),
        Dropout(0.2),
        Dense(16, activation="relu"),
        Dense(1)
    ])

    model.compile(
        optimizer="adam",
        loss="mean_squared_error"
    )

    early_stop = EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True
    )

    model.fit(
        X_train,
        y_train,
        epochs=30,
        batch_size=32,
        validation_split=0.1,
        shuffle=False,
        callbacks=[early_stop],
        verbose=1
    )

    predictions_scaled = model.predict(X_test, verbose=0)

    predictions = target_scaler.inverse_transform(
        predictions_scaled
    ).flatten()

    actual = target_scaler.inverse_transform(
        y_test
    ).flatten()

    mae = mean_absolute_error(actual, predictions)
    rmse = np.sqrt(mean_squared_error(actual, predictions))
    mape = np.mean(
        np.abs((actual - predictions) / actual)
    ) * 100

    print("MAE:", round(mae, 2))
    print("RMSE:", round(rmse, 2))
    print("MAPE:", round(mape, 2), "%")

    model.save(f"models/{company.lower()}_lstm.keras")

    joblib.dump(
        feature_scaler,
        f"models/{company.lower()}_lstm_feature_scaler.pkl"
    )

    joblib.dump(
        target_scaler,
        f"models/{company.lower()}_lstm_target_scaler.pkl"
    )

    lstm_results[company] = {
        "MAE": mae,
        "RMSE": rmse,
        "MAPE": mape
    }

print("\nAll LSTM models trained successfully!")



Training LSTM: TCS


C:\Users\mari sudha\AppData\Roaming\Python\Python313\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 7s 104ms/step - loss: 0.0221 - val_loss: 0.0321
Epoch 2/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 2s 82ms/step - loss: 0.0796 - val_loss: 0.1786
Epoch 3/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 2s 74ms/step - loss: 0.0773 - val_loss: 0.0692
Epoch 4/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 2s 74ms/step - loss: 0.0407 - val_loss: 0.0374
Epoch 5/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 2s 80ms/step - loss: 0.0181 - val_loss: 0.0131
Epoch 6/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 2s 76ms/step - loss: 0.0147 - val_loss: 0.0092
Epoch 7/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 2s 81ms/step - loss: 0.0137 - val_loss: 0.0142
Epoch 8/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 2s 63ms/step - loss: 0.0132 - val_loss: 0.0088
Epoch 9/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 2s 62ms/step - loss: 0.0114 - val_loss: 0.0080
Epoch 10/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 2s 62ms/step - loss: 0.0121 - val_loss: 0.0092
Epoch 11/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 2s 69ms/step - loss: 0.0107 - val_loss: 0.0081
Epoch 12/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 2s 90ms/step - loss: 0.

C:\Users\mari sudha\AppData\Roaming\Python\Python313\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


25/25 ━━━━━━━━━━━━━━━━━━━━ 6s 99ms/step - loss: 0.0389 - val_loss: 0.0345
Epoch 2/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 2s 79ms/step - loss: 0.0497 - val_loss: 0.0868
Epoch 3/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 2s 67ms/step - loss: 0.0476 - val_loss: 0.0366
Epoch 4/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 2s 75ms/step - loss: 0.0285 - val_loss: 0.0339
Epoch 5/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 2s 66ms/step - loss: 0.0236 - val_loss: 0.0245
Epoch 6/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 2s 63ms/step - loss: 0.0192 - val_loss: 0.0196
Epoch 7/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 2s 62ms/step - loss: 0.0151 - val_loss: 0.0059
Epoch 8/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 2s 85ms/step - loss: 0.0101 - val_loss: 0.0048
Epoch 9/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 2s 93ms/step - loss: 0.0095 - val_loss: 0.0067
Epoch 10/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 54ms/step - loss: 0.0100 - val_loss: 0.0083
Epoch 11/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 52ms/step - loss: 0.0106 - val_loss: 0.0068
Epoch 12/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 3s 62ms/step - loss: 0.0102 - val_l

C:\Users\mari sudha\AppData\Roaming\Python\Python313\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


25/25 ━━━━━━━━━━━━━━━━━━━━ 6s 97ms/step - loss: 0.0604 - val_loss: 0.0675
Epoch 2/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 2s 78ms/step - loss: 0.0540 - val_loss: 0.0094
Epoch 3/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 2s 82ms/step - loss: 0.0460 - val_loss: 0.0117
Epoch 4/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 2s 84ms/step - loss: 0.0204 - val_loss: 0.0108
Epoch 5/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 54ms/step - loss: 0.0146 - val_loss: 0.0080
Epoch 6/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 53ms/step - loss: 0.0138 - val_loss: 0.0064
Epoch 7/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 50ms/step - loss: 0.0120 - val_loss: 0.0066
Epoch 8/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 52ms/step - loss: 0.0110 - val_loss: 0.0066
Epoch 9/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 52ms/step - loss: 0.0099 - val_loss: 0.0066
Epoch 10/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 54ms/step - loss: 0.0109 - val_loss: 0.0067
Epoch 11/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 48ms/step - loss: 0.0105 - val_loss: 0.0070
MAE: 43.85
RMSE: 55.37
MAPE: 3.25 %

All LSTM models trained successfully!

In [32]:
pip install streamlit plotly

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [33]:

import os
import joblib
import numpy as np
import pandas as pd

from arch import arch_model

os.makedirs("models", exist_ok=True)

garch_results = {}

for company in ["TCS", "INFOSYS", "RELIANCE"]:

    print(f"\nTraining GARCH: {company}")

    df = feature_data[
        feature_data["Company"] == company
    ].copy()

    df = df.sort_values("Date")

    # Daily returns in percentage
    returns = (
        df["Close"].pct_change() * 100
    ).dropna()

    # Fit GARCH(1,1)
    model = arch_model(
        returns,
        mean="Zero",
        vol="GARCH",
        p=1,
        q=1,
        dist="normal",
        rescale=False
    )

    result = model.fit(disp="off")

    # Forecast next 5 trading days variance
    forecast = result.forecast(horizon=5)

    daily_variance = forecast.variance.iloc[-1].values

    five_day_volatility = np.sqrt(
        np.sum(daily_variance)
    )

    print("5-day forecast volatility:",
          round(five_day_volatility, 3), "%")

    print("Annualized volatility:",
          round(five_day_volatility * np.sqrt(252 / 5), 2), "%")

    joblib.dump(
        result,
        f"models/{company.lower()}_garch.pkl"
    )

    garch_results[company] = {
        "5_day_volatility_pct": five_day_volatility
    }

print("\nAll GARCH models trained successfully!")



Training GARCH: TCS
5-day forecast volatility: 4.074 %
Annualized volatility: 28.92 %

Training GARCH: INFOSYS
5-day forecast volatility: 3.668 %
Annualized volatility: 26.04 %

Training GARCH: RELIANCE
5-day forecast volatility: 4.924 %
Annualized volatility: 34.96 %

All GARCH models trained successfully!


In [34]:

import pandas as pd
import os

comparison = []

for company in ["TCS", "INFOSYS", "RELIANCE"]:

    comparison.append({
        "Company": company,
        "RF_Accuracy_%": round(
            rf_results[company] * 100, 2
        ),
        "XGB_MAE": round(
            xgb_results[company]["MAE"], 2
        ),
        "XGB_RMSE": round(
            xgb_results[company]["RMSE"], 2
        ),
        "XGB_MAPE_%": round(
            xgb_results[company]["MAPE"], 2
        ),
        "LSTM_MAE": round(
            lstm_results[company]["MAE"], 2
        ),
        "LSTM_RMSE": round(
            lstm_results[company]["RMSE"], 2
        ),
        "LSTM_MAPE_%": round(
            lstm_results[company]["MAPE"], 2
        ),
        "GARCH_5D_Volatility_%": round(
            garch_results[company]["5_day_volatility_pct"], 3
        )
    })

comparison_df = pd.DataFrame(comparison)

os.makedirs("outputs", exist_ok=True)

comparison_df.to_csv(
    "outputs/model_comparison.csv",
    index=False
)

display(comparison_df)

print("\nModel comparison report saved successfully!")


,Company,RF_Accuracy_%,XGB_MAE,XGB_RMSE,XGB_MAPE_%,LSTM_MAE,LSTM_RMSE,LSTM_MAPE_%,GARCH_5D_Volatility_%
0,TCS,44.30,354.68,409.32,15.61,379.17,433.64,16.66,4.074
1,INFOSYS,51.90,72.07,88.24,6.33,55.79,72.92,4.80,3.668
2,RELIANCE,48.52,44.08,55.53,3.22,43.85,55.37,3.25,4.924



Model comparison report saved successfully!
